In [10]:
print("hello")

hello


In [6]:
!pip install yfinance --upgrade -q
import yfinance as yf
import pandas as pd


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [7]:
def fetch_stock_data(ticker: str, period: str = "2y") -> pd.DataFrame:
    """
    Fetches historical OHLCV data for a given ticker.
    
    ticker: NSE ticker with .NS suffix e.g. "RELIANCE.NS"
    period: how far back to go — "1y", "2y", "5y"
    
    returns: cleaned dataframe with OHLCV columns, no timezone, no missing values
    """
    try:
        t = yf.Ticker(ticker)
        df = t.history(period=period)
        
        # if yfinance returns empty dataframe, ticker doesn't exist
        if df.empty:
            raise ValueError(f"No data found for ticker: {ticker}")
        
        # clean
        df = df.drop(columns=['Dividends', 'Stock Splits'])
        df.index = df.index.tz_localize(None)
        
        # forward fill any gaps — if a day is missing, use previous day's value
        # this handles rare yfinance gaps without breaking indicator calculations
        df = df.ffill()
        
        return df
    
    except Exception as e:
        print(f"Error fetching {ticker}: {e}")
        return pd.DataFrame()  # return empty df, backend will handle the error


# test it
reliance = fetch_stock_data("RELIANCE.NS")
tcs = fetch_stock_data("TCS.NS")
fake = fetch_stock_data("FAKEFAKE.NS")  # should handle gracefully

print("Reliance shape:", reliance.shape)
print("TCS shape:", tcs.shape)
print("Fake ticker:", fake.shape)  # should be (0, 0)

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: FAKEFAKE.NS"}}}
$FAKEFAKE.NS: possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")


Error fetching FAKEFAKE.NS: No data found for ticker: FAKEFAKE.NS
Reliance shape: (498, 5)
TCS shape: (498, 5)
Fake ticker: (0, 0)


In [8]:
def fetch_stock_info(ticker: str) -> dict:
    """
    Fetches company overview data for the dashboard header card.
    
    returns: dict with name, current price, % change, volume, 52w high/low
    """
    try:
        t = yf.Ticker(ticker)
        info = t.info
        
        # .info returns a massive dictionary — you only need these fields
        overview = {
            "ticker": ticker,
            "name": info.get("longName", ticker),
            "current_price": info.get("currentPrice") or info.get("regularMarketPrice"),
            "previous_close": info.get("previousClose"),
            "percent_change": None,  # calculate below
            "volume": info.get("volume") or info.get("regularMarketVolume"),
            "week_52_high": info.get("fiftyTwoWeekHigh"),
            "week_52_low": info.get("fiftyTwoWeekLow"),
            "market_cap": info.get("marketCap"),
            "pe_ratio": info.get("trailingPE"),
        }
        
        # calculate % change from previous close
        if overview["current_price"] and overview["previous_close"]:
            change = overview["current_price"] - overview["previous_close"]
            overview["percent_change"] = round((change / overview["previous_close"]) * 100, 2)
        
        return overview
    
    except Exception as e:
        print(f"Error fetching info for {ticker}: {e}")
        return {}


# test it
info = fetch_stock_info("RELIANCE.NS")
for key, value in info.items():
    print(f"{key}: {value}")
    

ticker: RELIANCE.NS
name: Reliance Industries Limited
current_price: 1354.5
previous_close: 1349.6
percent_change: 0.36
volume: 7104959
week_52_high: 1611.8
week_52_low: 1290.0
market_cap: 18329733431296
pe_ratio: 22.677048


In [9]:
!pip install feedparser -q

import feedparser

def fetch_headlines(company_name: str) -> list:
    """
    Fetches headlines from Google News RSS — much more relevant than NewsAPI free tier.
    No API key needed.
    """
    try:
        # Google News RSS URL for a search query
        query = company_name.replace(" ", "+")
        url = f"https://news.google.com/rss/search?q={query}&hl=en-IN&gl=IN&ceid=IN:en"
        
        feed = feedparser.parse(url)
        
        
        if not feed.entries:
            return []
        BLOCKED_SOURCES = ["facebook.com", "twitter.com", "instagram.com", "reddit.com"]

        headlines = [
    {
        "title": entry.title,
        "published_at": entry.published,
        "source": entry.source.get("title", "Google News") if hasattr(entry, "source") else "Google News"
    }
    for entry in feed.entries[:15]
    if not any(
        blocked in (entry.source.get("title", "") if hasattr(entry, "source") else "").lower()
        for blocked in BLOCKED_SOURCES
    )
]
        
        return headlines
    
    except Exception as e:
        print(f"Error fetching news: {e}")
        return []


# test
headlines = fetch_headlines("Reliance Industries")
print(f"Got {len(headlines)} headlines\n")
for h in headlines:
    print(f"[{h['source']}] {h['title']}")



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Got 14 headlines

[The Economic Times] Reliance Industries, TCS, among 10 stocks with sharpest decrease in retail shareholding in Q4 - The Economic Times
[Upstox] Reliance Industries' stock jumps over 2.5% amid weak trade; possible triggers explained - Upstox
[Mint] Reliance Industries share price rises 2.5% despite stock market crash. Do you own? - Mint
[Markets Mojo] Reliance Industries Ltd Sees High Value Trading Amid Mixed Technical Signals - Markets Mojo
[TradingView] Reliance Industries Says Voluntary Strike Off Of BAM DLR Kolkata Pvt Ltd - TradingView
[ET Telecom] Andhra Pradesh clears over 800 acres for Reliance's ₹1 lakh-cr AI data centre in Vizianagaram - ET Telecom
[The New Indian Express] Reliance Industries, two firms tell SC they will write to Centre for mediation in KG-basin dispute - The New Indian Express
[Telegraph India] Reliance bribery probe: Executive get